# 01 — Data Exploration

This notebook explores the pretraining corpus and SFT datasets:
- Token count and vocabulary coverage
- BPE tokenisation visualisation
- Length distribution of training examples
- SFT dataset category breakdown

In [ ]:
import sys, os
sys.path.insert(0, '..')   # run from notebooks/ directory

import json
import matplotlib.pyplot as plt
import collections

from src.tokenizer.bpe_tokenizer import BPETokenizer
from src.tokenizer.vocab import describe_vocabulary

tokenizer = BPETokenizer()
print(tokenizer)
print()
print(describe_vocabulary())

## 1. Pretraining corpus stats

In [ ]:
with open('../data/pretrain/data.txt', encoding='utf-8') as f:
    pretrain_text = f.read()

token_ids = tokenizer.encode(pretrain_text)

print(f'Characters   : {len(pretrain_text):,}')
print(f'Words        : {len(pretrain_text.split()):,}')
print(f'Tokens       : {len(token_ids):,}')
print(f'Unique tokens: {len(set(token_ids)):,} / {tokenizer.vocab_size:,} vocab')
print(f'Vocab coverage: {tokenizer.vocabulary_coverage(pretrain_text)*100:.1f}%')
print(f'Avg chars/token: {tokenizer.average_token_length(pretrain_text):.2f}')

## 2. BPE tokenisation visualisation

In [ ]:
sample_sentences = [
    'The lighthouse keeper climbed the spiral stairs.',
    'She discovered the letter hidden in her mother\'s coat.',
    'The detective examined the empty frame on the wall.',
    'A small act of kindness can change everything.',
]

for sent in sample_sentences:
    tokens = tokenizer.encode(sent)
    print(f'\nText   : {sent}')
    print(f'Tokens : {len(tokens)}')
    print(f'Repr   : {tokenizer.show_tokenisation(sent)}')

## 3. Token frequency distribution

In [ ]:
freq = tokenizer.token_frequency(pretrain_text)
top_20 = sorted(freq.items(), key=lambda x: -x[1])[:20]

token_strs = [repr(tokenizer.decode([tid])) for tid, _ in top_20]
counts     = [cnt for _, cnt in top_20]

plt.figure(figsize=(12, 4))
plt.bar(range(20), counts, color='#2563eb')
plt.xticks(range(20), token_strs, rotation=45, ha='right', fontsize=9)
plt.title('Top 20 Most Frequent Tokens (Pretraining Corpus)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('../results/plots/token_frequency.png', dpi=150)
plt.show()
print('Saved → results/plots/token_frequency.png')

## 4. Training example length distribution

In [ ]:
# Paragraph-level lengths (approximate training examples)
paragraphs = [p.strip() for p in pretrain_text.split('\n\n') if p.strip()]
lengths    = [tokenizer.token_count(p) for p in paragraphs]

plt.figure(figsize=(8, 4))
plt.hist(lengths, bins=30, color='#2563eb', edgecolor='white', alpha=0.8)
plt.axvline(x=256, color='#dc2626', linestyle='--', label='Context length (256)')
plt.xlabel('Tokens per paragraph')
plt.ylabel('Count')
plt.title('Paragraph Length Distribution')
plt.legend()
plt.tight_layout()
plt.savefig('../results/plots/length_distribution.png', dpi=150)
plt.show()

print(f'Paragraphs: {len(paragraphs)}')
print(f'Avg length: {sum(lengths)/len(lengths):.0f} tokens')
print(f'Max length: {max(lengths)} tokens')
print(f'% > 256   : {100*sum(1 for l in lengths if l > 256)/len(lengths):.1f}%  (will be windowed)')

## 5. SFT dataset exploration

In [ ]:
# Load both SFT categories
story_data  = json.load(open('../data/finetune/story_completion/data.json'))
poetry_data = json.load(open('../data/finetune/poetry/data.json'))
all_sft     = story_data + poetry_data

print(f'Story completion entries : {len(story_data)}')
print(f'Poetry entries           : {len(poetry_data)}')
print(f'Total SFT entries        : {len(all_sft)}')

In [ ]:
# Output length distribution
output_lengths = [tokenizer.token_count(e['output']) for e in all_sft]

plt.figure(figsize=(8, 4))
plt.hist(output_lengths, bins=20, color='#7c3aed', edgecolor='white', alpha=0.8)
plt.xlabel('Output length (tokens)')
plt.ylabel('Count')
plt.title('SFT Output Length Distribution')
plt.tight_layout()
plt.savefig('../results/plots/sft_output_lengths.png', dpi=150)
plt.show()

print(f'Average output: {sum(output_lengths)/len(output_lengths):.0f} tokens')
print(f'Min output    : {min(output_lengths)} tokens')
print(f'Max output    : {max(output_lengths)} tokens')

In [ ]:
# Show a few examples
import random
random.seed(42)
samples = random.sample(all_sft, 3)

for i, entry in enumerate(samples, 1):
    print(f'\n─── Example {i} ─────────────────────────────────')
    print(f'Instruction: {entry["instruction"]}')
    print(f'Output (first 200 chars): {entry["output"][:200]}…')